# WESAD CGAP: Full LOSO Ablation Study (3 Seeds)

This Google Colab notebook runs a **15-participant leave-one-subject-out (LOSO) evaluation** for the proposed CGAP model, with three random seeds. It performs inner participant-grouped hyperparameter tuning for full CGAP and evaluates controlled ablations using the selected parameters.

## Models evaluated
1. **CGAP Full** — ExtraTrees global model + class-weighted online SGD + probability blending + sequential labeled-feedback updates.
2. **Global only** — ExtraTrees global branch only.
3. **Online only** — class-weighted SGD branch with sequential updates.
4. **CGAP no online updates** — both branches and tuned blend, but online model is frozen during test.
5. **CGAP no class weights** — same structure, but the online SGD branch is trained and updated without class weights.

**Protocol note:** after each test window is predicted, its true label is used to update online variants. This is a *labeled-feedback replay* evaluation, not an unlabeled deployment setting. Hyperparameters are tuned on inner training participants only. Ablations reuse the full CGAP's tuned hyperparameters to isolate component effects and keep runtime manageable. This is a controlled ablation design; it does not independently optimize every ablation variant.


## 1. Mount Google Drive and load participant windows

Expected location: `MyDrive/DataCon/Project 6 Wesad/S1_windows.csv` through `S15_windows.csv`. Edit `DATA_DIR` if your files are stored elsewhere.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re, json, time, copy, os
import numpy as np
import pandas as pd

DATA_DIR = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
subject_files = sorted(
    p for p in DATA_DIR.glob('S*_windows.csv')
    if re.fullmatch(r'S\d+_windows', p.stem)
)
if len(subject_files) != 15:
    raise FileNotFoundError(
        f'Expected 15 corrected participant CSVs in {DATA_DIR}; found {len(subject_files)}. '
        'Check DATA_DIR and filenames.'
    )

df = pd.concat((pd.read_csv(path) for path in subject_files), ignore_index=True)
print('Participant files:', len(subject_files), '| rows:', len(df), '| participants:', df['subject'].nunique())
display(df.head())


## 2. Validate data and separate features, labels, and subject groups

The identifiers and labels are excluded from the feature matrix. This cell checks participant coverage, label names, ordering, duplicates, and numeric feature values.

In [ ]:
required = {'subject', 'window', 'label', 'label_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'

df = df.copy()
df['subject'] = df['subject'].astype(str)
df['label'] = df['label'].astype(str)
df = df.sort_values(['subject', 'window']).reset_index(drop=True)

FEATURES = [c for c in df.columns if c not in required]
assert FEATURES, 'No sensor feature columns found'
df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='coerce')
assert df[FEATURES].notna().all().all(), 'Missing feature values found; inspect input files.'
assert np.isfinite(df[FEATURES].to_numpy(dtype=float)).all(), 'Infinite feature values found.'
assert df.duplicated().sum() == 0, 'Duplicate rows found.'
assert df.groupby('subject')['window'].apply(lambda s: s.is_monotonic_increasing).all(), 'Windows are not ordered.'
CLASSES = ['baseline', 'stress', 'amusement']
assert set(df['label'].unique()) == set(CLASSES), f'Unexpected labels: {sorted(df.label.unique())}'
assert df['subject'].nunique() == 15, f'Expected 15 participants; got {df.subject.nunique()}'

df = df.reset_index(drop=True)
X = df[FEATURES].astype(float)
y = df['label'].astype(str)
groups = df['subject'].astype(str)
SUBJECTS = sorted(groups.unique())
print('Rows:', len(df), '| features:', len(FEATURES), '| subjects:', len(SUBJECTS))
print('Label counts:', y.value_counts().to_dict())


## 3. Define CGAP, tuning, and ablation evaluation

The tuning search is intentionally compact for Colab: 2 tree settings × 2 SGD regularization values × 3 blend weights, evaluated on two participant-grouped inner splits. Set `INNER_SPLITS = 3` or expand grids for a more exhaustive study if runtime permits.

In [ ]:
from itertools import product
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score

SEEDS = [17, 42, 2026]
OUTER = LeaveOneGroupOut()
INNER_SPLITS = 2
CHUNK_SIZE = 10

TREE_GRID = [(0.8, 2), (1.0, 5)]  # (max_features, min_samples_leaf)
SGD_ALPHA_GRID = [0.0001, 0.001]
BLEND_GRID = [0.25, 0.50, 0.75]

MODEL_NAMES = [
    'CGAP Full',
    'Global only',
    'Online only',
    'CGAP no online updates',
    'CGAP no class weights',
]
METRIC_COLUMNS = ['accuracy', 'balanced_accuracy', 'macro_f1', 'macro_precision', 'macro_recall']

def score(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
    }

def class_weights_for(y_train):
    counts = y_train.value_counts()
    missing = set(CLASSES) - set(counts.index)
    if missing:
        raise ValueError(f'Training partition is missing classes: {missing}. Use group splits containing all classes.')
    return {c: len(y_train) / (len(CLASSES) * counts[c]) for c in CLASSES}

def make_global(tree_params, seed):
    max_features, min_leaf = tree_params
    return make_pipeline(
        SimpleImputer(strategy='median'),
        ExtraTreesClassifier(
            n_estimators=80,
            max_features=max_features,
            min_samples_leaf=min_leaf,
            class_weight='balanced',
            n_jobs=-1,
            random_state=seed,
        ),
    )

def make_scaler():
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler())

def make_online(alpha, seed):
    return SGDClassifier(
        loss='log_loss',
        alpha=alpha,
        class_weight=None,
        max_iter=1500,
        tol=1e-3,
        random_state=seed,
    )

def fit_branches(train_positions, tree_params, sgd_alpha, seed):
    train_x = X.iloc[train_positions]
    train_y = y.iloc[train_positions]

    global_model = make_global(tree_params, seed)
    global_model.fit(train_x, train_y)

    scaler = make_scaler()
    train_scaled = scaler.fit_transform(train_x)
    weights = class_weights_for(train_y)

    online_weighted = make_online(sgd_alpha, seed)
    sample_weights = np.asarray([weights[label] for label in train_y])
    online_weighted.fit(train_scaled, train_y, sample_weight=sample_weights)

    online_unweighted = make_online(sgd_alpha, seed)
    online_unweighted.fit(train_scaled, train_y)

    return global_model, scaler, online_weighted, online_unweighted, weights

def ordered_positions(frame):
    return frame.sort_values(['subject', 'window']).index.to_numpy()

def proba_in_class_order(model, x, classes=CLASSES):
    p_local = model.predict_proba(x)[0]
    model_classes = list(model.classes_)
    return np.asarray([p_local[model_classes.index(c)] for c in classes], dtype=float)

def evaluate_one_subject(global_model, scaler, online_model, weights, blend, frame,
                         variant, use_global=True, use_online=True,
                         update_online=True, use_weights=True, chunk_size=CHUNK_SIZE):
    positions = ordered_positions(frame)
    actual, predicted, curve = [], [], []

    for count, pos in enumerate(positions, start=1):
        x_raw = X.loc[[pos]]
        x_scaled = scaler.transform(x_raw)
        true_label = y.loc[pos]

        p_global = proba_in_class_order(global_model, x_raw) if use_global else None
        p_online = proba_in_class_order(online_model, x_scaled) if use_online else None

        if use_global and use_online:
            probabilities = (1.0 - blend) * p_global + blend * p_online
        elif use_global:
            probabilities = p_global
        elif use_online:
            probabilities = p_online
        else:
            raise ValueError('At least one branch must be enabled.')

        pred = CLASSES[int(np.argmax(probabilities))]
        actual.append(true_label)
        predicted.append(pred)

        # Strict order: predict first, then reveal label and optionally update.
        if use_online and update_online:
            if use_weights:
                sample_weight = np.asarray([weights[true_label]])
                online_model.partial_fit(
                    x_scaled, np.asarray([true_label]),
                    classes=np.asarray(CLASSES), sample_weight=sample_weight
                )
            else:
                online_model.partial_fit(
                    x_scaled, np.asarray([true_label]),
                    classes=np.asarray(CLASSES)
                )

        if count % chunk_size == 0 or count == len(positions):
            curve.append({'windows_seen': count, **score(actual, predicted)})

    return actual, predicted, curve

def tune_cgap(train_positions, seed):
    # Inner split is participant-grouped; only training participants are used here.
    train_frame = df.iloc[train_positions]
    inner = GroupShuffleSplit(
        n_splits=INNER_SPLITS, test_size=0.25, random_state=seed
    )
    candidates = []
    for tree_params, alpha, blend in product(TREE_GRID, SGD_ALPHA_GRID, BLEND_GRID):
        split_scores = []
        for fit_local, val_local in inner.split(
            train_frame, groups=train_frame['subject'].astype(str)
        ):
            fit_positions = train_frame.iloc[fit_local].index.to_numpy()
            val_frame = train_frame.iloc[val_local].copy()
            gmodel, scaler, online_w, _, weights = fit_branches(
                fit_positions, tree_params, alpha, seed
            )

            # Reset the online state for each validation participant. No adaptation
            # state is carried from one participant to another.
            participant_scores = []
            for val_subject in sorted(val_frame['subject'].astype(str).unique()):
                subject_frame = val_frame[val_frame['subject'].astype(str) == val_subject]
                online_copy = copy.deepcopy(online_w)
                actual, pred, _ = evaluate_one_subject(
                    gmodel, scaler, online_copy, weights, blend, subject_frame,
                    variant='inner-CGAP', use_global=True, use_online=True,
                    update_online=True, use_weights=True, chunk_size=10**9
                )
                participant_scores.append(
                    f1_score(actual, pred, labels=CLASSES, average='macro', zero_division=0)
                )
            split_scores.append(float(np.mean(participant_scores)))

        candidates.append((
            float(np.mean(split_scores)), tree_params, alpha, blend
        ))

    candidates.sort(key=lambda row: row[0], reverse=True)
    best_score, best_tree, best_alpha, best_blend = candidates[0]
    return best_score, best_tree, best_alpha, best_blend

print('Model variants:', MODEL_NAMES)
print('Tuning configurations per outer fold:', len(TREE_GRID) * len(SGD_ALPHA_GRID) * len(BLEND_GRID))


## 4. Run full 15-subject LOSO ablation with 3 seeds and resumable checkpoints

Each outer fold holds out one participant. Hyperparameters are tuned only on the other participants. All five ablation variants use the same selected full-CGAP parameters within a seed/fold. The notebook checkpoints immediately after each completed seed/fold.

In [ ]:
CHECKPOINT_DIR = DATA_DIR / 'cgap_ablation_checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILE = CHECKPOINT_DIR / 'ablation_results_checkpoint.csv'
SELECTED_FILE = CHECKPOINT_DIR / 'ablation_selected_parameters_checkpoint.csv'
CURVE_FILE = CHECKPOINT_DIR / 'ablation_learning_curve_checkpoint.csv'

results = pd.read_csv(RESULTS_FILE) if RESULTS_FILE.exists() else pd.DataFrame()
selected_parameters = pd.read_csv(SELECTED_FILE) if SELECTED_FILE.exists() else pd.DataFrame()
learning_curve = pd.read_csv(CURVE_FILE) if CURVE_FILE.exists() else pd.DataFrame()

completed = set()
if not results.empty and {'seed', 'fold', 'model'}.issubset(results.columns):
    completed = set(
        zip(
            results.loc[results['model'].eq('CGAP Full'), 'seed'].astype(int),
            results.loc[results['model'].eq('CGAP Full'), 'fold'].astype(int),
        )
    )
print('Completed seed/fold pairs:', len(completed), sorted(completed)[:10])

def atomic_csv(frame, path):
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp_path, index=False)
    tmp_path.replace(path)

def save_all_checkpoints():
    atomic_csv(results, RESULTS_FILE)
    atomic_csv(selected_parameters, SELECTED_FILE)
    atomic_csv(learning_curve, CURVE_FILE)
    print('Checkpoint saved.', flush=True)

for seed in SEEDS:
    seed = int(seed)
    print(f'\n========== SEED {seed} ==========', flush=True)

    for fold, (train_pos, test_pos) in enumerate(OUTER.split(X, y, groups=groups), start=1):
        key = (seed, int(fold))
        if key in completed:
            print(f'Skip completed seed={seed}, fold={fold}', flush=True)
            continue

        start_time = time.time()
        train_subjects = set(groups.iloc[train_pos])
        test_subjects = set(groups.iloc[test_pos])
        assert len(test_subjects) == 1 and train_subjects.isdisjoint(test_subjects)
        held_out = next(iter(test_subjects))
        test_frame = df.iloc[test_pos].copy()

        print(f'Seed={seed} | fold={fold}/15 | held-out={held_out} | train rows={len(train_pos)} | test rows={len(test_pos)}', flush=True)

        # Hyperparameter tuning is performed only for the full CGAP model.
        inner_score, tree_params, sgd_alpha, blend = tune_cgap(train_pos, seed)
        print(f'  Tuned inner macro-F1={inner_score:.4f}; tree={tree_params}; alpha={sgd_alpha}; blend={blend}', flush=True)

        global_model, scaler, online_weighted, online_unweighted, weights = fit_branches(
            train_pos, tree_params, sgd_alpha, seed
        )

        # Independent copies prevent online adaptation from leaking between variants.
        variants = [
            ('CGAP Full', True, True, True, True, blend, copy.deepcopy(online_weighted)),
            ('Global only', True, False, False, True, 0.0, None),
            ('Online only', False, True, True, True, 1.0, copy.deepcopy(online_weighted)),
            ('CGAP no online updates', True, True, False, True, blend, copy.deepcopy(online_weighted)),
            ('CGAP no class weights', True, True, True, False, blend, copy.deepcopy(online_unweighted)),
        ]

        fold_records, fold_selected, fold_curve = [], [], []
        tuned_params = {
            'tree_max_features': tree_params[0],
            'tree_min_samples_leaf': tree_params[1],
            'sgd_alpha': sgd_alpha,
            'online_blend_weight': blend,
        }

        for model_name, use_global, use_online, update_online, use_weights, model_blend, online_copy in variants:
            actual, prediction, curve = evaluate_one_subject(
                global_model, scaler, online_copy, weights, model_blend, test_frame,
                variant=model_name, use_global=use_global, use_online=use_online,
                update_online=update_online, use_weights=use_weights, chunk_size=CHUNK_SIZE
            )
            fold_records.append({
                'seed': seed, 'fold': fold, 'held_out_subject': held_out,
                'model': model_name, 'best_inner_macro_f1': inner_score,
                'best_params': json.dumps(tuned_params),
                **score(actual, prediction),
            })
            fold_selected.append({
                'seed': seed, 'fold': fold, 'held_out_subject': held_out,
                'model': model_name, 'inner_macro_f1': inner_score,
                'best_params': json.dumps(tuned_params),
            })
            for point in curve:
                fold_curve.append({
                    'seed': seed, 'fold': fold, 'held_out_subject': held_out,
                    'model': model_name, **point
                })

        # Append and persist all data for this fold before moving on.
        results = pd.concat([results, pd.DataFrame(fold_records)], ignore_index=True)
        selected_parameters = pd.concat([selected_parameters, pd.DataFrame(fold_selected)], ignore_index=True)
        learning_curve = pd.concat([learning_curve, pd.DataFrame(fold_curve)], ignore_index=True)
        save_all_checkpoints()
        completed.add(key)

        full_row = next(r for r in fold_records if r['model'] == 'CGAP Full')
        print(
            f'Completed seed={seed}, fold={fold}, subject={held_out}, '
            f'CGAP macro-F1={full_row["macro_f1"]:.4f}, '
            f'elapsed={time.time()-start_time:.1f}s',
            flush=True
        )

print('\nCompleted seed/fold pairs:', len(completed), '| expected:', len(SEEDS) * 15)
print('Result rows:', len(results), '| expected:', len(SEEDS) * 15 * len(MODEL_NAMES))
display(results.tail(10))


## 5. Summarize ablation results

First average the three seed runs for each held-out participant. Then summarize across the 15 participants. This prevents participants with more rows from dominating the final summary.

In [ ]:
METRIC_COLUMNS = ['accuracy', 'balanced_accuracy', 'macro_f1', 'macro_precision', 'macro_recall']

per_subject = (
    results.groupby(['model', 'held_out_subject'], as_index=False)[METRIC_COLUMNS]
    .mean()
)
summary = (
    per_subject.groupby('model')[METRIC_COLUMNS]
    .agg(['mean', 'std', 'min', 'max'])
    .round(4)
)
display(summary)

print('Participant-level macro-F1 by model:')
display(
    per_subject.pivot(index='held_out_subject', columns='model', values='macro_f1')
    .round(3)
)

print('Tuned hyperparameter frequencies:')
display(
    selected_parameters[selected_parameters['model'].eq('CGAP Full')]
    .groupby('best_params').size()
    .rename('seed_fold_count').reset_index()
    .sort_values('seed_fold_count', ascending=False)
)


## 6. Plot mean macro-F1 and learning curves

In [ ]:
import matplotlib.pyplot as plt

macro = per_subject.groupby('model')['macro_f1'].agg(['mean', 'std']).sort_values('mean')
plt.figure(figsize=(10, 5))
plt.barh(macro.index, macro['mean'], xerr=macro['std'])
plt.xlabel('Mean participant-level macro-F1 (error bars: SD across participants)')
plt.title('CGAP ablation study — 3-seed LOSO')
plt.tight_layout()
plt.show()

if not learning_curve.empty:
    full_curve = learning_curve[learning_curve['model'].isin(MODEL_NAMES)]
    curve_summary = (
        full_curve.groupby(['model', 'windows_seen'])['macro_f1']
        .mean().reset_index()
    )
    plt.figure(figsize=(10, 6))
    for model_name in MODEL_NAMES:
        sub = curve_summary[curve_summary['model'].eq(model_name)]
        if not sub.empty:
            plt.plot(sub['windows_seen'], sub['macro_f1'], marker='o', label=model_name)
    plt.xlabel('Held-out participant windows processed')
    plt.ylabel('Cumulative macro-F1')
    plt.title('Cumulative test performance by ablation')
    plt.legend()
    plt.tight_layout()
    plt.show()


## 7. Export final CSV files to Google Drive

In [ ]:
OUTPUT_DIR = DATA_DIR / 'cgap_ablation_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv(OUTPUT_DIR / 'cgap_ablation_loso_3seeds_raw.csv', index=False)
per_subject.to_csv(OUTPUT_DIR / 'cgap_ablation_loso_per_subject.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'cgap_ablation_loso_summary.csv')
selected_parameters.to_csv(OUTPUT_DIR / 'cgap_ablation_selected_parameters.csv', index=False)
learning_curve.to_csv(OUTPUT_DIR / 'cgap_ablation_learning_curves.csv', index=False)

print('Saved all outputs to:', OUTPUT_DIR)
print('Checkpoint folder:', CHECKPOINT_DIR)


## Reporting and reproducibility notes

- Expected completed evaluations: **3 seeds × 15 held-out participants × 5 variants = 225 model evaluations**.
- Tuning is nested within each outer training partition. The held-out participant is not used for hyperparameter selection.
- Ablations reuse full-CGAP tuned hyperparameters for controlled component comparisons; they are not independently tuned.
- The online variants use true labels to update after prediction. Report this explicitly as labeled-feedback adaptation.
- For a strict no-label-at-test deployment claim, remove the `partial_fit` updates and design an unlabeled adaptation method.
- If Colab disconnects, rerun the notebook from the beginning. Completed seed/fold pairs are skipped using Drive checkpoints.
- Check the final result count and all 15 held-out subjects before reporting results. The notebook has not been executed against your Drive data in this chat, so no performance values are claimed here.
